# A/B Testing & Retention Analysis

## Опис проєкту

У проєкті проведено аналіз результатів A/B тестування в мобільній грі Cookie Cats для оцінки впливу змін у продукті на утримання користувачів.

Було виконано статистичний аналіз retention metrics, перевірку статистичних гіпотез та оцінку довірчих інтервалів для контрольної та тестової груп.

### Формування статистичних гіпотез

У грі Cookie Cats перші ворота були перенесені з 30-го рівня (`gate_30`) на 40-й рівень (`gate_40`) для оцінки впливу цієї зміни на утримання користувачів.

* `gate_30` — контрольна група
* `gate_40` — тестова група

Оскільки заздалегідь невідомо, чи вплине зміна позитивно чи негативно на retention, використовується двосторонній статистичний тест:

$$H_0: p_1 = p_2$$
$$H_a: p_1 \ne p_2$$

де:

$p_1$ - частка утримання гравців у контрольній групі gate_30

$p_2$ - частка утримання гравців у тестовій групі gate_40

Рівень статистичної значущості:
$$\alpha = 0.05$$

### Розрахунок необхідного розміру вибірки

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [2]:
effect_size = sms.proportion_effectsize(0.19, 0.20)

In [3]:
effect_size

np.float64(-0.025241594409087353)

In [4]:
required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )

required_n = ceil(required_n)

print(f"Users per group: {required_n}")
print(f"Total users needed: {required_n * 2}")

Users per group: 24638
Total users needed: 49276


Для виявлення збільшення retention з 19% до 20% при рівні значущості 0.05 та потужності 80% необхідно приблизно 24638 користувачів у кожній групі. Загальна кількість користувачів становить близько 49276.

### Завантаження та первинний огляд даних

На цьому етапі завантажуються дані A/B-тестування та виконується первинний аналіз retention-показників для контрольної (`gate_30`) та тестової (`gate_40`) груп.

In [5]:
df = pd.read_csv('../statistical_hypothesis/cookie_cats.csv')

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [7]:
df.groupby('version')['retention_7'].mean()

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

Середнє значення retention_7 показує частку користувачів, які повернулися до гри через 7 днів після встановлення.

Оскільки ми не знаємо наперед, яка версія дає краще утримання, використовуємо двосторонній тест:

$$H_0: p_1 = p_2$$
$$H_a: p_1 \ne p_2$$

де:

$p_1$ - частка користувачів, які повернулися через 7 днів у версії gate_30

$p_2$ - частка користувачів, які повернулися через 7 днів у версії gate_40

### Перевірка статистичної значущості результатів

Для оцінки впливу зміни в грі на показник `retention_7` було використано двосторонній z-test для двох пропорцій.

Також для контрольної (`gate_30`) та тестової (`gate_40`) груп були розраховані 95% довірчі інтервали.

У межах аналізу необхідно визначити:

* чи є різниця між групами статистично значущою;
* чи перетинаються довірчі інтервали двох груп;
* як зміна вплинула на утримання користувачів.

In [8]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [9]:
control_results = df[df['version'] == 'gate_30']['retention_7']
treatment_results = df[df['version'] == 'gate_40']['retention_7']

In [10]:
n_con = control_results.count()
n_treat = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_con, n_treat]

In [11]:
successes

[np.int64(8502), np.int64(8279)]

In [12]:
z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat:.3f}, {upper_treat:.3f}]')

z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи control: [0.187, 0.194]
Довірчий інтервал 95% для групи treatment: [0.178, 0.186]


### Висновок

Оскільки p-value = 0.002 є меншою за рівень значущості (\alpha = 0.05), нульову гіпотезу (H_0) відхиляємо. Це свідчить про наявність статистично значущої різниці між двома версіями гри.

95% довірчі інтервали для груп `gate_30` та `gate_40` не перетинаються, що додатково підтверджує статистично значущу різницю між групами.

Результати аналізу показують, що версія `gate_30` демонструє вищий рівень утримання користувачів порівняно з `gate_40`.

Отже, перенесення воріт із 30-го на 40-й рівень негативно вплинуло на retention користувачів.

### Аналіз залежності між версією гри та retention

Для перевірки наявності статистичної залежності між версією гри та показником утримання користувачів на 7-й день було використано тест Xi-квадрат (χ²).

У межах аналізу:

* сформульовано статистичні гіпотези;
* виконано перевірку статистичної значущості;
* проаналізовано результати тесту та p-value.

**Гіпотези:**

* $H_0$: між версією гри та retention немає статистично значущої залежності.
  
* $H_a$: між версією гри та retention існує статистично значуща залежність.

In [13]:
from scipy.stats import chi2_contingency

In [14]:
contingency_table = pd.crosstab(df['version'], df['retention_7'])

In [15]:
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


### Висновок

Оскільки p-value є меншою за рівень значущості (\alpha = 0.05), нульову гіпотезу відхиляємо.

Результати χ²-тесту свідчать про наявність статистично значущої залежності між версією гри та retention користувачів на 7-й день.

Отже, зміна розташування воріт із 30-го на 40-й рівень вплинула на поведінку користувачів та рівень їх утримання.